# 11 · GKP states in position space

Gottesman–Kitaev–Preskill (GKP) states encode a qubit in a grid of peaks spaced by $\sqrt\pi$
(Gottesman, Kitaev, Preskill, *PRA* **64**, 012310, 2001). Real GKP states have peaks of finite width
$\Delta$ under a Gaussian envelope; at 10–15 dB of squeezing they need hundreds of Fock levels, and
their Fock amplitudes oscillate — a hard case for the Fock representation, and an easy one in
position space.

1. The design measurement behind the CV extension: quantics ranks in the Fock and position bases.
2. The Wigner function of a GKP state (a lattice of positive and negative peaks).
3. Logical readout by homodyne binning, and its error under Gaussian displacement noise,
   against the classic estimate of GKP's original paper.
4. Loss as displacement noise: loss $\eta$ followed by amplification $1/\eta$ is a pure Gaussian
   displacement channel of variance $(1-\eta)/\eta$ (Albert *et al.*, *PRA* **97**, 032346, 2018) — checked
   with the Gaussian backend's channels.
5. Exact bit-by-bit sampling of homodyne outcomes from the tensor train.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
import tutorial_utils as tu
from tutorial_utils import cd

print(tu.cv_banner())
if tu.HAVE_CV:
    from qttlind.cv import QuadratureGrid, grids as gr, measure as me
    from qttlind.bosons.states import from_function
if tu.HAVE_GAUSSIAN:
    from qttlind.gaussian import ops
    from qttlind.gaussian.ops import check_channel
checks = {}

def self_dual(bits):
    N = 2**bits; L = np.sqrt(2 * np.pi * N); dx = L / N
    return (np.arange(N) - N / 2) * dx, dx


def odd_bin_weight(x):
    # 1 inside the bins of odd multiples of sqrt(pi), 1/2 exactly on a bin edge (trapezoidal rule),
    # 0 elsewhere. On a 13-bit self-dual grid dx = sqrt(pi)/64: the edges are grid points.
    u = np.asarray(x) / np.sqrt(np.pi)
    on_edge = np.abs(np.abs(u - np.floor(u)) - 0.5) < 1e-9
    k = np.rint(u).astype(int)
    return np.where(on_edge, 0.5, (np.mod(k, 2) == 1).astype(float))

## 1. Ranks: Fock versus position

Same TT-SVD rule as everywhere (discarded weight $10^{-16}$ per bond). The Fock amplitudes come
from the dense Hermite transform of the wavefunction, truncated where the tail drops below $10^{-12}$.

In [ ]:
def fock_rank(psi, x, dx, nmax=2048):
    c = tu.hermite_functions(nmax, x) @ psi * dx
    c = c / np.linalg.norm(c)
    tail = np.cumsum((np.abs(c) ** 2)[::-1])[::-1]
    cut = int(np.argmax(tail < 1e-12))
    Rn = max(1, int(np.ceil(np.log2(cut))))
    return Rn, max(tu.tt_ranks(tu.tt_svd(c[: 2**Rn], Rn, tol=1e-16)))

rows = []
x, dx = self_dual(13)
for delta in (0.4, 0.3, 0.2, 0.15):
    psi = tu.gkp_wavefunction(x, delta)
    Rn, rf = fock_rank(psi, x, dx)
    if tu.HAVE_CV:
        rx = gr.gkp(delta, QuadratureGrid(13), tol=1e-16).max_rank
    else:
        rx = max(tu.tt_ranks(tu.tt_svd(psi * np.sqrt(dx), 13, tol=1e-16)))
    rows.append((delta, -10 * np.log10(2 * delta**2), Rn, rf, rx))
print(" delta   dB   Fock bits  Fock rank  position rank (13 bits)")
for d, db, Rn, rf, rx in rows:
    print(f"{d:6.2f} {db:5.1f} {Rn:8d} {rf:10d} {rx:12d}")

## 2. Wigner function

$W(x,p)=\frac1\pi\int dy\,\overline{\psi(x+y)}\,\psi(x-y)\,e^{2ipy}$, from the position-space
wavefunction (the tensor train when `"cv"` is registered).

In [ ]:
x, dx = self_dual(11)
delta = 0.3
psi = tu.gkp_wavefunction(x, delta)
if tu.HAVE_CV:
    g = QuadratureGrid(11)
    psi_tt = gr.wavefunction(gr.gkp(delta, g), g)
    checks["GKP wavefunction: cv.grids vs dense helper"] = (np.abs(np.abs(psi_tt) - np.abs(psi)).max(), 1e-8)
xs_out = x[(x > -5) & (x < 5)][::3]
ps = np.linspace(-5, 5, 121)
W = tu.wigner_from_wavefunction(psi, x, ps, xs_out)
fig, ax = plt.subplots(figsize=(5, 4.5))
lim = np.abs(W).max()
ax.contourf(xs_out, ps, W, 41, cmap="RdBu_r", vmin=-lim, vmax=lim)
ax.set_xlabel("x"); ax.set_ylabel("p"); ax.set_aspect("equal"); ax.set_title(f"GKP |0>, Delta = {delta}")
tu.savefig(fig, "11_gkp_wigner.png")
print("integral of W:", W.sum() * (xs_out[1] - xs_out[0]) * (ps[1] - ps[0]), "(the plotted window cuts the tails)")

## 3. Homodyne readout and displacement noise

Measure $x$ and decide "0" if $x$ is closer to an even multiple of $\sqrt\pi$. (Numerical detail: on
this 13-bit self-dual grid $dx=\sqrt\pi/64$, so the bin edges are grid points and get weight 1/2 —
the trapezoidal rule. Counting them fully on one side would bias the error by ~10 %.) A random displacement
of variance $\sigma^2$ in $x$ convolves the homodyne distribution with a Gaussian. GKP's estimate for
ideal peaks: an error occurs when a peak (variance $\Delta^2/2+\sigma^2$ for $|\psi|^2$) crosses $\pm\sqrt\pi/2$,

$$
P_{\rm err}\approx \sum_{k\ \rm odd}\Big[\Phi\big(\tfrac{(k+\frac12)\sqrt\pi}{s}\big)-\Phi\big(\tfrac{(k-\frac12)\sqrt\pi}{s}\big)\Big]
\approx \mathrm{erfc}\Big(\frac{\sqrt\pi}{2\sqrt2\,s}\Big),\qquad s^2=\frac{\Delta^2}{2}+\sigma^2 .
$$

In [ ]:
x, dx = self_dual(13)
delta = 0.2
psi = tu.gkp_wavefunction(x, delta)
P = np.abs(psi) ** 2 * dx
wrong = odd_bin_weight(x)

def error_with_noise(P, sigma):
    if sigma == 0:
        return P @ wrong
    kern = np.exp(-((np.arange(-400, 401) * dx) ** 2) / (2 * sigma**2)); kern /= kern.sum()
    return np.convolve(P, kern, mode="same") @ wrong

sigmas = np.linspace(0.0, 0.48, 17)
numeric = [error_with_noise(P, s) for s in sigmas]
estimate = [erfc(np.sqrt(np.pi) / (2 * np.sqrt(2) * np.sqrt(delta**2 / 2 + s**2))) for s in sigmas]
if tu.HAVE_CV:
    g = QuadratureGrid(13)
    x0 = -g.L / 2
    bins = from_function(lambda j: odd_bin_weight(x0 + j * g.dx).astype(complex), 13, 1e-20)
    p_tt = me.interval_probability(gr.gkp(delta, g), bins)
    checks["GKP error (sigma=0): tensor train vs dense"] = (abs(p_tt - numeric[0]), 1e-9)
checks["GKP error vs GKP estimate (sigma = 0.3, relative)"] = (abs(numeric[10] - estimate[10]) / estimate[10], 0.01)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(sigmas, numeric, "o", label="binning the exact distribution")
ax.semilogy(sigmas, estimate, "-", label="erfc estimate (GKP 2001)")
ax.set_xlabel("displacement noise sigma"); ax.set_ylabel("logical error"); ax.set_ylim(1e-12, 1); ax.legend()
tu.savefig(fig, "11_gkp_error.png");

## 4. Loss is displacement noise (after amplification)

Loss $\eta$ ($X=\sqrt\eta$, $Y=(1-\eta)/2$) followed by a phase-insensitive amplifier of gain
$1/\eta$ ($X=\eta^{-1/2}$, $Y=(1/\eta-1)/2$) composes to $X=I$, $Y=\frac{1-\eta}{\eta}I$: random displacements
of variance $\sigma^2=(1-\eta)/\eta$ per quadrature. With the Gaussian backend this is checked on the
covariance of any state.

In [ ]:
etas = np.linspace(0.85, 0.999, 12)
err = [error_with_noise(P, np.sqrt((1 - e) / e)) for e in etas]
if tu.HAVE_GAUSSIAN:
    from qttlind.gaussian import GaussianState
    s = GaussianState.squeezed_vacuum(0.7, 0.4)
    e = 0.9
    composed = ops.amplifier(ops.loss(s, e), 1 / e)
    direct = ops.thermal_noise(s, (1 - e) / e)
    checks["loss + amplifier = displacement noise (covariance)"] = (np.abs(composed.cov - direct.cov).max(), 1e-12)
    check_channel(np.eye(2), (1 - e) / e * np.eye(2))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(1 - etas, err, "o-")
ax.set_xlabel("loss 1 - eta"); ax.set_ylabel("logical error after amplification"); ax.set_title(f"Delta = {delta}")
tu.savefig(fig, "11_gkp_loss.png");

## 5. Sampling homodyne outcomes bit by bit

`cv.measure.sample` draws the most significant bit first (which half of the box), then the next,
down to the grid spacing: exact samples in $O(R\chi^2)$ each.

In [ ]:
if tu.HAVE_CV:
    g = QuadratureGrid(12)
    rng = np.random.default_rng(1)
    psi_tt = gr.gkp(0.25, g)
    samples = me.sample(psi_tt, 4000, rng)
    xs = (samples - g.size / 2) * g.dx
    xg = g.x(); dens = np.abs(gr.wavefunction(psi_tt, g)) ** 2
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(xs, bins=240, range=(-12, 12), density=True, alpha=0.6, label="4000 samples")
    ax.plot(xg, dens, lw=0.8, label="|psi(x)|^2")
    ax.set_xlim(-12, 12); ax.set_xlabel("x"); ax.legend(); tu.savefig(fig, "11_gkp_samples.png")
    frac_even = 1 - np.mean(odd_bin_weight(xs))
    checks["sampled fraction of even bins (expect ~1)"] = (abs(1 - frac_even), 0.01)
else:
    print("qttlind.cv is not available: sampling skipped.")

## Acceptance table

In [ ]:
for name, (dev, tol) in checks.items():
    print(f"{'ok ' if dev <= tol else 'FAIL'} {name:<52s} deviation {dev:9.2e}  (tolerance {tol:.0e})")
assert all(dev <= tol for dev, tol in checks.values()), "a benchmark is outside its tolerance"